# Pokémon Type Classifier
Multi-label classifier using official artwork from `pokemon_data.csv`.

## 1. Imports & Setup

In [ ]:
import os
import csv
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image
import requests
from io import BytesIO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

## 2. Pokémon Types

In [ ]:
POKEMON_TYPES = [
    "normal", "fire", "water", "grass", "electric", "ice",
    "fighting", "poison", "ground", "flying", "psychic",
    "bug", "rock", "ghost", "dark", "dragon", "steel", "fairy"
]

## 3. Dataset Class

In [ ]:
class PokemonDataset(Dataset):
    SPRITE_KEYS = ['front_default', 'front_shiny', 'back_default', 'back_shiny', 'official_artwork']

    def __init__(self, csv_file, transform=None, img_dir='pokemon_images'):
        self.transform = transform
        self.img_dir = img_dir
        os.makedirs(img_dir, exist_ok=True)

        # generate list of all sprites
        self.items = []
        with open(csv_file) as f:
            rows = list(csv.DictReader(f))
            for row in rows:
                for i, key in enumerate(self.SPRITE_KEYS):
                    url = row.get(key)
                    if url:  # only include if URL exists
                        self.items.append({
                            'id': row['id'],
                            'types': row['types'],
                            'url': url,
                            'sprite_type': key,
                            'sprite_name': f"{row['id']}_{key}.png"
                        })

    def load_sprite(self, url, local_name):
        filepath = os.path.join(self.img_dir, local_name)
        try:
            if not os.path.exists(filepath):
                resp = requests.get(url, timeout=5)
                resp.raise_for_status()
                with open(filepath, 'wb') as f:
                    f.write(resp.content)
            img = Image.open(filepath).convert('RGB')
            return img, False  # False indicates not a placeholder
        except:
            # return placeholder if needed
            img = Image.new('RGB', (224, 224), (0, 0, 0))
            return img, True  # True indicates placeholder

    def encode_types(self, type_string):
        labels = type_string.split(", ")
        return torch.tensor([1 if t in labels else 0 for t in POKEMON_TYPES], dtype=torch.float32)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        image, placeholder = self.load_sprite(item['url'], item['sprite_name'])
        if self.transform:
            image = self.transform(image)

        return {
            'image': image,
            'types': self.encode_types(item['types']),
            'sprite_type': item['sprite_type'],
            'placeholder': placeholder
        }


## 4. Data Transforms & Dataset Split

In [ ]:
from torchvision.models import ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
transform = weights.transforms()

dataset = PokemonDataset('pokemon_data.csv', transform)

# split the dataset here into a 4:1 ratio
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

## 5. Model Definition

In [ ]:
def build_model():
    weights = ResNet50_Weights.DEFAULT
    model = resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, len(POKEMON_TYPES))
    return model

model = build_model().to(DEVICE)

## 6. Training Loop

In [ ]:
import glob

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
EPOCHS = 100
checkpoint_dir = 'checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# check for latest checkpoint
checkpoint_files = glob.glob(os.path.join(checkpoint_dir, 'checkpoint_epoch_*.pt'))
latest_checkpoint = None
start_epoch = 0

if checkpoint_files:
    latest_checkpoint = max(checkpoint_files, key=lambda x: int(x.split('_')[-1].split('.')[0]))
    print(f"Resuming from checkpoint: {latest_checkpoint}")
    checkpoint = torch.load(latest_checkpoint, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']

for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss = 0
    for batch in train_loader:
        imgs = batch['image'].to(DEVICE)
        labels = batch['types'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {running_loss:.4f}")

    # save the checkpoint
    checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pt')
    torch.save({
        'epoch': epoch+1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': running_loss,
    }, checkpoint_path)
    print(f"Saved checkpoint: {checkpoint_path}")

    # log losses for graphing purposes later
    with open("loss_log.txt", "a") as file:
        file.write(f"Epoch {epoch} loss: {running_loss}\n")

    # stop training if loss is below threshold
    if running_loss <= 0.2:
        print(f"Stopping training early at epoch {epoch+1} as loss reached {running_loss:.4f}")
        break

## 7. Evaluation on Test Set

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        imgs = batch['image'].to(DEVICE)
        labels = batch['types'].to(DEVICE)
        logits = model(imgs)
        probs = torch.sigmoid(logits)
        all_preds.append(probs)
        all_labels.append(labels)

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

# display per-type results
pred_labels = (all_preds > 0.5).float()
accuracy_per_type = (pred_labels == all_labels).float().mean(dim=0)

for t, acc in zip(POKEMON_TYPES, accuracy_per_type):
    print(f"{t:<10}: {acc:.3f}")

## 8. Prediction / Inference

In [ ]:
import os
from PIL import Image
import torch
from torchvision import transforms

# Find latest checkpoint
ckpt_dir = 'checkpoints'
all_ckpts = [f for f in os.listdir(ckpt_dir) if f.endswith('.pt')]
latest_ckpt = max(all_ckpts, key=lambda x: int(x.split('_epoch_')[1].split('.pt')[0]))
ckpt_path = os.path.join(ckpt_dir, latest_ckpt)
print(f"Loading checkpoint: {ckpt_path}")

# Load model from checkpoint
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.to(DEVICE)
model.eval()

def predict(model, image, threshold=0.3):
    """Predict Pokémon types from a PIL Image."""
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                             std=[0.229, 0.224, 0.225])
    ])
    
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    img_tensor = transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(img_tensor)
        probs = torch.sigmoid(logits)[0]
    
    results = [(t, float(p)) for t, p in zip(POKEMON_TYPES, probs) if p > threshold]
    if not results:
        max_idx = torch.argmax(probs).item()
        results = [(POKEMON_TYPES[max_idx], float(probs[max_idx]))]
    return results

# make guess for every image in folder
image_dir = 'test_images'
all_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.avif'))]

for img_file in all_files:
    img_path = os.path.join(image_dir, img_file)
    img = Image.open(img_path)
    predicted = predict(model, img)
    print(f"{img_file}: {predicted}")